# Notebook 02 — Fine-Tuning con QLoRA
**Objetivo:** Entrenar Qwen2.5-7B-Instruct especializado en contratos legales
**Entradas:** `/content/drive/MyDrive/agente-contratos-cto/dataset/contratos_sft.jsonl`
**Salidas:** `/content/drive/MyDrive/agente-contratos-cto/adapter/`
**Requisito:** Entorno de ejecución → Cambiar tipo de entorno → GPU T4
**Tiempo estimado:** 2-3 horas

In [ ]:
!pip install -q torch==2.3.0 transformers==4.44.0 peft==0.12.0 trl==0.10.1 bitsandbytes==0.43.3 datasets==2.21.0 accelerate==0.33.0 sentencepiece==0.2.0

In [ ]:
import torch
assert torch.cuda.is_available(), "⚠️ Activar GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU T4"
print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
print(f"VRAM disponible: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
import json
from typing import Dict, List, Any
from datasets import Dataset, DatasetDict
from google.colab import drive

# Montar Google Drive para acceder al dataset y guardar resultados
drive.mount('/content/drive')

# --- Rutas de archivos ---
RUTA_DATASET = "/content/drive/MyDrive/agente-contratos-cto/dataset/contratos_sft.jsonl"


def cargar_dataset_jsonl(ruta_archivo: str) -> List[Dict[str, Any]]:
    """Carga un archivo JSONL y devuelve una lista de diccionarios.

    Args:
        ruta_archivo: Ruta absoluta al archivo JSONL a cargar.

    Returns:
        Lista de diccionarios con los ejemplos del dataset.
    """
    ejemplos: List[Dict[str, Any]] = []
    with open(ruta_archivo, "r", encoding="utf-8") as archivo:
        for linea in archivo:
            linea = linea.strip()
            if linea:
                ejemplos.append(json.loads(linea))
    return ejemplos


def dividir_dataset(
    ejemplos: List[Dict[str, Any]],
    porcentaje_entrenamiento: float = 0.9,
    semilla: int = 42,
) -> DatasetDict:
    """Divide una lista de ejemplos en conjuntos de entrenamiento y evaluación.

    Args:
        ejemplos: Lista de diccionarios con los datos.
        porcentaje_entrenamiento: Fracción de datos para entrenamiento (0-1).
        semilla: Semilla para reproducibilidad.

    Returns:
        DatasetDict con las particiones 'entrenamiento' y 'evaluacion'.
    """
    dataset_completo = Dataset.from_list(ejemplos)
    particiones = dataset_completo.train_test_split(
        test_size=1.0 - porcentaje_entrenamiento,
        seed=semilla,
    )
    return DatasetDict({
        "entrenamiento": particiones["train"],
        "evaluacion": particiones["test"],
    })


# --- Cargar y dividir el dataset ---
ejemplos_crudos = cargar_dataset_jsonl(RUTA_DATASET)
print(f"Total de ejemplos cargados: {len(ejemplos_crudos)}")

dataset = dividir_dataset(ejemplos_crudos, porcentaje_entrenamiento=0.9)
print(f"Ejemplos de entrenamiento: {len(dataset['entrenamiento'])}")
print(f"Ejemplos de evaluación: {len(dataset['evaluacion'])}")
print(f"\nEjemplo de muestra:")
print(json.dumps(dataset["entrenamiento"][0], indent=2, ensure_ascii=False))

## Configuración del Modelo y QLoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- Configuración de cuantización en 4 bits ---
NOMBRE_MODELO = "Qwen/Qwen2.5-7B-Instruct"

configuracion_bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


def cargar_modelo_cuantizado(nombre_modelo: str, config_bnb: BitsAndBytesConfig) -> tuple:
    """Carga el modelo base con cuantización de 4 bits y su tokenizador.

    Args:
        nombre_modelo: Identificador del modelo en Hugging Face Hub.
        config_bnb: Configuración de BitsAndBytes para cuantización.

    Returns:
        Tupla con (modelo_cuantizado, tokenizador).
    """
    tokenizador = AutoTokenizer.from_pretrained(
        nombre_modelo,
        trust_remote_code=True,
    )
    # Asegurar que el token de relleno esté configurado
    if tokenizador.pad_token is None:
        tokenizador.pad_token = tokenizador.eos_token

    modelo = AutoModelForCausalLM.from_pretrained(
        nombre_modelo,
        quantization_config=config_bnb,
        device_map="auto",
        trust_remote_code=True,
    )
    modelo.config.use_cache = False

    return modelo, tokenizador


# --- Cargar modelo y tokenizador ---
print(f"Cargando modelo: {NOMBRE_MODELO}")
print("Esto puede tardar varios minutos...")
modelo, tokenizador = cargar_modelo_cuantizado(NOMBRE_MODELO, configuracion_bnb)
print(f"Modelo cargado exitosamente en: {modelo.device}")
print(f"Vocabulario del tokenizador: {len(tokenizador)} tokens")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# --- Configuración de LoRA ---
CONFIG_LORA = {
    "r": 16,
    "lora_alpha": 32,
    "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": "CAUSAL_LM"
}


def aplicar_lora_al_modelo(
    modelo: AutoModelForCausalLM,
    config_lora: dict,
) -> AutoModelForCausalLM:
    """Prepara el modelo para entrenamiento en baja precisión y aplica adaptadores LoRA.

    Args:
        modelo: Modelo base cuantizado.
        config_lora: Diccionario con los hiperparámetros de LoRA.

    Returns:
        Modelo con adaptadores LoRA aplicados, listo para entrenamiento.
    """
    # Preparar el modelo para entrenamiento con cuantización k-bit
    modelo = prepare_model_for_kbit_training(modelo)

    # Crear configuración de LoRA
    configuracion_lora = LoraConfig(**config_lora)

    # Aplicar adaptadores LoRA al modelo
    modelo_con_lora = get_peft_model(modelo, configuracion_lora)

    return modelo_con_lora


def mostrar_parametros_entrenables(modelo: AutoModelForCausalLM) -> None:
    """Muestra el número de parámetros entrenables vs totales del modelo.

    Args:
        modelo: Modelo con adaptadores LoRA aplicados.
    """
    parametros_entrenables = 0
    parametros_totales = 0
    for _, parametro in modelo.named_parameters():
        parametros_totales += parametro.numel()
        if parametro.requires_grad:
            parametros_entrenables += parametro.numel()

    porcentaje = 100 * parametros_entrenables / parametros_totales
    print(f"Parámetros entrenables: {parametros_entrenables:,}")
    print(f"Parámetros totales:     {parametros_totales:,}")
    print(f"Porcentaje entrenable:  {porcentaje:.2f}%")


# --- Aplicar LoRA al modelo ---
modelo = aplicar_lora_al_modelo(modelo, CONFIG_LORA)
mostrar_parametros_entrenables(modelo)

## Preparación de Datos para SFT

In [ ]:
from typing import Optional

# --- Mensaje de sistema para el agente de contratos ---
MENSAJE_SISTEMA = "Eres un agente experto en análisis de contratos tecnológicos para CTOs."


def formatear_ejemplo_chat(ejemplo: Dict[str, Any]) -> str:
    """Convierte un ejemplo del dataset al formato de plantilla de chat de Qwen2.5.

    Construye una conversación con tres turnos: sistema, usuario y asistente,
    usando la plantilla de chat nativa del tokenizador.

    Args:
        ejemplo: Diccionario con las claves 'instruccion', 'entrada' y 'salida'.

    Returns:
        Cadena formateada con la plantilla de chat aplicada.
    """
    instruccion: str = ejemplo.get("instruccion", "")
    entrada: Optional[str] = ejemplo.get("entrada", "")
    salida: str = ejemplo.get("salida", "")

    # Construir el mensaje del usuario combinando instrucción y entrada
    if entrada and entrada.strip():
        mensaje_usuario = f"{instruccion}\n\n{entrada}"
    else:
        mensaje_usuario = instruccion

    # Construir la conversación en formato de mensajes
    mensajes = [
        {"role": "system", "content": MENSAJE_SISTEMA},
        {"role": "user", "content": mensaje_usuario},
        {"role": "assistant", "content": salida},
    ]

    # Aplicar la plantilla de chat del tokenizador
    texto_formateado = tokenizador.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=False,
    )

    return texto_formateado


# --- Verificar el formato con un ejemplo ---
ejemplo_muestra = dataset["entrenamiento"][0]
texto_formateado = formatear_ejemplo_chat(ejemplo_muestra)
print("=" * 60)
print("EJEMPLO FORMATEADO PARA SFT:")
print("=" * 60)
print(texto_formateado)
print("=" * 60)
print(f"Longitud en tokens: {len(tokenizador.encode(texto_formateado))}")

## Entrenamiento con SFTTrainer

In [ ]:
import os
from transformers import TrainingArguments
from trl import SFTTrainer

# --- Configuración de entrenamiento ---
CONFIG_ENTRENAMIENTO = {
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "gradient_checkpointing": True,
    "fp16": True,
    "save_strategy": "epoch",
    "evaluation_strategy": "epoch",
    "load_best_model_at_end": True,
    "output_dir": "/content/drive/MyDrive/agente-contratos-cto/adapter",
    "logging_steps": 10,
    "report_to": "none"
}

# --- Manejo de reanudación tras caída de sesión ---
RUTA_ADAPTER = "/content/drive/MyDrive/agente-contratos-cto/adapter"
reanudar_desde = RUTA_ADAPTER if os.path.exists(f"{RUTA_ADAPTER}/trainer_state.json") else None

if reanudar_desde:
    print(f"Se detectó un checkpoint previo en: {reanudar_desde}")
    print("El entrenamiento se reanudará desde el último punto de control.")
else:
    print("No se encontró checkpoint previo. Se iniciará el entrenamiento desde cero.")


def crear_argumentos_entrenamiento(config: dict) -> TrainingArguments:
    """Crea los argumentos de entrenamiento a partir del diccionario de configuración.

    Args:
        config: Diccionario con los hiperparámetros de entrenamiento.

    Returns:
        Objeto TrainingArguments configurado para SFTTrainer.
    """
    return TrainingArguments(**config)


def crear_entrenador(
    modelo: AutoModelForCausalLM,
    tokenizador: AutoTokenizer,
    argumentos: TrainingArguments,
    dataset_entrenamiento: Dataset,
    dataset_evaluacion: Dataset,
    funcion_formato: callable,
) -> SFTTrainer:
    """Crea y configura el entrenador SFT con todos los componentes necesarios.

    Args:
        modelo: Modelo con adaptadores LoRA aplicados.
        tokenizador: Tokenizador del modelo.
        argumentos: Argumentos de entrenamiento configurados.
        dataset_entrenamiento: Conjunto de datos para entrenamiento.
        dataset_evaluacion: Conjunto de datos para evaluación.
        funcion_formato: Función que formatea cada ejemplo al formato de chat.

    Returns:
        Instancia de SFTTrainer lista para ejecutar el entrenamiento.
    """
    return SFTTrainer(
        model=modelo,
        tokenizer=tokenizador,
        args=argumentos,
        train_dataset=dataset_entrenamiento,
        eval_dataset=dataset_evaluacion,
        formatting_func=funcion_formato,
        max_seq_length=2048,
        packing=False,
    )


# --- Crear y ejecutar el entrenador ---
argumentos_entrenamiento = crear_argumentos_entrenamiento(CONFIG_ENTRENAMIENTO)

entrenador = crear_entrenador(
    modelo=modelo,
    tokenizador=tokenizador,
    argumentos=argumentos_entrenamiento,
    dataset_entrenamiento=dataset["entrenamiento"],
    dataset_evaluacion=dataset["evaluacion"],
    funcion_formato=formatear_ejemplo_chat,
)

print("Iniciando entrenamiento...")
print(f"Épocas: {CONFIG_ENTRENAMIENTO['num_train_epochs']}")
print(f"Tamaño de lote efectivo: {CONFIG_ENTRENAMIENTO['per_device_train_batch_size'] * CONFIG_ENTRENAMIENTO['gradient_accumulation_steps']}")
print(f"Tasa de aprendizaje: {CONFIG_ENTRENAMIENTO['learning_rate']}")
print("-" * 60)

resultado_entrenamiento = entrenador.train(resume_from_checkpoint=reanudar_desde)

print("-" * 60)
print("Entrenamiento completado exitosamente.")
print(f"Pérdida final de entrenamiento: {resultado_entrenamiento.training_loss:.4f}")

## Guardar Adapter y Métricas

In [ ]:
import json
import os

# --- Rutas de salida ---
RUTA_ADAPTER = "/content/drive/MyDrive/agente-contratos-cto/adapter"
RUTA_METRICAS = "/content/drive/MyDrive/agente-contratos-cto/metricas"
RUTA_PERDIDA = os.path.join(RUTA_METRICAS, "perdida_entrenamiento.json")


def guardar_adapter_y_tokenizador(
    modelo: AutoModelForCausalLM,
    tokenizador: AutoTokenizer,
    ruta_salida: str,
) -> None:
    """Guarda el adaptador LoRA y el tokenizador en la ruta especificada.

    Args:
        modelo: Modelo con adaptadores LoRA entrenados.
        tokenizador: Tokenizador del modelo.
        ruta_salida: Ruta del directorio donde guardar los archivos.
    """
    os.makedirs(ruta_salida, exist_ok=True)
    modelo.save_pretrained(ruta_salida)
    tokenizador.save_pretrained(ruta_salida)
    print(f"Adaptador LoRA guardado en: {ruta_salida}")
    print(f"Tokenizador guardado en: {ruta_salida}")


def extraer_historial_perdida(entrenador: SFTTrainer) -> Dict[str, List[float]]:
    """Extrae el historial de pérdida de entrenamiento y evaluación del entrenador.

    Args:
        entrenador: Instancia de SFTTrainer con el entrenamiento completado.

    Returns:
        Diccionario con las listas de pérdida de entrenamiento y evaluación,
        junto con los pasos correspondientes.
    """
    historial = {
        "perdida_entrenamiento": [],
        "pasos_entrenamiento": [],
        "perdida_evaluacion": [],
        "epocas_evaluacion": [],
    }

    for registro in entrenador.state.log_history:
        if "loss" in registro:
            historial["perdida_entrenamiento"].append(registro["loss"])
            historial["pasos_entrenamiento"].append(registro.get("step", 0))
        if "eval_loss" in registro:
            historial["perdida_evaluacion"].append(registro["eval_loss"])
            historial["epocas_evaluacion"].append(registro.get("epoch", 0))

    return historial


def guardar_metricas(historial: Dict[str, List[float]], ruta_archivo: str) -> None:
    """Guarda el historial de métricas en un archivo JSON.

    Args:
        historial: Diccionario con el historial de pérdidas.
        ruta_archivo: Ruta completa del archivo JSON de salida.
    """
    directorio = os.path.dirname(ruta_archivo)
    os.makedirs(directorio, exist_ok=True)

    with open(ruta_archivo, "w", encoding="utf-8") as archivo:
        json.dump(historial, archivo, indent=2, ensure_ascii=False)

    print(f"Métricas guardadas en: {ruta_archivo}")


# --- Guardar adaptador y tokenizador ---
guardar_adapter_y_tokenizador(modelo, tokenizador, RUTA_ADAPTER)

# --- Extraer y guardar historial de pérdida ---
historial_perdida = extraer_historial_perdida(entrenador)
guardar_metricas(historial_perdida, RUTA_PERDIDA)

print(f"\nResumen de métricas:")
print(f"  Registros de pérdida de entrenamiento: {len(historial_perdida['perdida_entrenamiento'])}")
print(f"  Registros de pérdida de evaluación:    {len(historial_perdida['perdida_evaluacion'])}")

## Visualización de la Curva de Pérdida

In [ ]:
import json
import matplotlib.pyplot as plt

# --- Ruta del archivo de métricas ---
RUTA_PERDIDA = "/content/drive/MyDrive/agente-contratos-cto/metricas/perdida_entrenamiento.json"


def cargar_metricas(ruta_archivo: str) -> Dict[str, List[float]]:
    """Carga el historial de métricas desde un archivo JSON.

    Args:
        ruta_archivo: Ruta completa al archivo JSON con las métricas.

    Returns:
        Diccionario con el historial de pérdidas.
    """
    with open(ruta_archivo, "r", encoding="utf-8") as archivo:
        return json.load(archivo)


def graficar_curva_perdida(historial: Dict[str, List[float]]) -> None:
    """Genera un gráfico con las curvas de pérdida de entrenamiento y evaluación.

    Args:
        historial: Diccionario con las listas de pérdida y pasos/épocas.
    """
    fig, (eje_izq, eje_der) = plt.subplots(1, 2, figsize=(14, 5))

    # --- Gráfico de pérdida de entrenamiento ---
    eje_izq.plot(
        historial["pasos_entrenamiento"],
        historial["perdida_entrenamiento"],
        color="#2196F3",
        linewidth=1.5,
        label="Pérdida de entrenamiento",
    )
    eje_izq.set_xlabel("Pasos de entrenamiento", fontsize=12)
    eje_izq.set_ylabel("Pérdida", fontsize=12)
    eje_izq.set_title("Curva de Pérdida — Entrenamiento", fontsize=13)
    eje_izq.legend(fontsize=10)
    eje_izq.grid(True, alpha=0.3)

    # --- Gráfico de pérdida de evaluación ---
    eje_der.plot(
        historial["epocas_evaluacion"],
        historial["perdida_evaluacion"],
        color="#FF5722",
        linewidth=2,
        marker="o",
        markersize=8,
        label="Pérdida de evaluación",
    )
    eje_der.set_xlabel("Época", fontsize=12)
    eje_der.set_ylabel("Pérdida", fontsize=12)
    eje_der.set_title("Curva de Pérdida — Evaluación", fontsize=13)
    eje_der.legend(fontsize=10)
    eje_der.grid(True, alpha=0.3)

    plt.suptitle("Métricas de Entrenamiento — QLoRA Fine-Tuning", fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    # --- Resumen numérico ---
    print(f"\nPérdida inicial de entrenamiento: {historial['perdida_entrenamiento'][0]:.4f}")
    print(f"Pérdida final de entrenamiento:   {historial['perdida_entrenamiento'][-1]:.4f}")
    if historial["perdida_evaluacion"]:
        print(f"Mejor pérdida de evaluación:      {min(historial['perdida_evaluacion']):.4f}")


# --- Cargar y graficar ---
historial_perdida = cargar_metricas(RUTA_PERDIDA)
graficar_curva_perdida(historial_perdida)

## Prueba Rápida del Modelo Fine-Tuneado

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- Rutas ---
NOMBRE_MODELO_BASE = "Qwen/Qwen2.5-7B-Instruct"
RUTA_ADAPTER = "/content/drive/MyDrive/agente-contratos-cto/adapter"

# --- Mensaje de sistema ---
MENSAJE_SISTEMA = "Eres un agente experto en análisis de contratos tecnológicos para CTOs."

# --- Cláusula de prueba ---
CLAUSULA_PRUEBA = (
    "Cláusula 7.3 — Limitación de Responsabilidad: "
    "El proveedor no será responsable por daños indirectos, incidentales, "
    "especiales o consecuentes, incluyendo pero no limitándose a la pérdida "
    "de beneficios, datos o uso, incluso si el proveedor ha sido advertido "
    "de la posibilidad de dichos daños. La responsabilidad total acumulada "
    "del proveedor bajo este contrato no excederá el monto total pagado por "
    "el cliente durante los últimos doce (12) meses."
)


def cargar_modelo_con_adapter(
    nombre_modelo_base: str,
    ruta_adapter: str,
) -> tuple:
    """Carga el modelo base con cuantización y aplica el adaptador LoRA entrenado.

    Args:
        nombre_modelo_base: Identificador del modelo base en Hugging Face Hub.
        ruta_adapter: Ruta al directorio con el adaptador LoRA guardado.

    Returns:
        Tupla con (modelo_con_adapter, tokenizador).
    """
    configuracion_bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    tokenizador = AutoTokenizer.from_pretrained(ruta_adapter, trust_remote_code=True)

    modelo_base = AutoModelForCausalLM.from_pretrained(
        nombre_modelo_base,
        quantization_config=configuracion_bnb,
        device_map="auto",
        trust_remote_code=True,
    )

    modelo_con_adapter = PeftModel.from_pretrained(modelo_base, ruta_adapter)
    modelo_con_adapter.eval()

    return modelo_con_adapter, tokenizador


def generar_respuesta(
    modelo: AutoModelForCausalLM,
    tokenizador: AutoTokenizer,
    consulta_usuario: str,
    mensaje_sistema: str,
    max_nuevos_tokens: int = 512,
    temperatura: float = 0.7,
) -> str:
    """Genera una respuesta del modelo fine-tuneado dada una consulta del usuario.

    Args:
        modelo: Modelo con adaptador LoRA cargado.
        tokenizador: Tokenizador del modelo.
        consulta_usuario: Texto de la consulta del usuario.
        mensaje_sistema: Mensaje de sistema para el contexto.
        max_nuevos_tokens: Número máximo de tokens a generar.
        temperatura: Temperatura de muestreo para la generación.

    Returns:
        Texto generado por el modelo como respuesta.
    """
    mensajes = [
        {"role": "system", "content": mensaje_sistema},
        {"role": "user", "content": consulta_usuario},
    ]

    texto_entrada = tokenizador.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True,
    )

    tokens_entrada = tokenizador(
        texto_entrada,
        return_tensors="pt",
    ).to(modelo.device)

    with torch.no_grad():
        tokens_salida = modelo.generate(
            **tokens_entrada,
            max_new_tokens=max_nuevos_tokens,
            temperature=temperatura,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
        )

    # Decodificar solo los tokens nuevos generados
    longitud_entrada = tokens_entrada["input_ids"].shape[1]
    tokens_generados = tokens_salida[0][longitud_entrada:]
    respuesta = tokenizador.decode(tokens_generados, skip_special_tokens=True)

    return respuesta


# --- Cargar modelo con adaptador ---
print("Cargando modelo con adaptador LoRA...")
modelo_ft, tokenizador_ft = cargar_modelo_con_adapter(NOMBRE_MODELO_BASE, RUTA_ADAPTER)
print("Modelo cargado exitosamente.\n")

# --- Ejecutar prueba ---
consulta = f"Analiza la siguiente cláusula contractual e identifica los riesgos para el CTO:\n\n{CLAUSULA_PRUEBA}"

print("=" * 60)
print("CONSULTA:")
print("=" * 60)
print(consulta)
print("\n" + "=" * 60)
print("RESPUESTA DEL MODELO FINE-TUNEADO:")
print("=" * 60)

respuesta = generar_respuesta(
    modelo=modelo_ft,
    tokenizador=tokenizador_ft,
    consulta_usuario=consulta,
    mensaje_sistema=MENSAJE_SISTEMA,
)

print(respuesta)